# Synaptic Routing Architecture (SRA)
## 11: シナプスの動的削除（Hot-Swapの取り消し・特定ドメインのパージ）

このノートブックでは、SRAの機能である「シナプスの削除」を実証します。具体的には以下の2つのシナリオを検証します。
1. **プラグインの取り外し (`pop_synapses`)**: 後から追加（Hot-Swap）したシナプスを末尾から物理的に削除し、追加前の状態に復元します。
2. **特定ドメインのパージ (`clear_synapses`)**: 初期のベースモデルから特定の機能（例：Math）だけを抽出し、他と共用していないシナプスを安全に「ゼロクリア（空きスロット化）」します。

※ このノートブックはGPUがなくても実行可能です。

In [ ]:
import sys
import os
import torch
import tiktoken

# SRAライブラリへのパスを通す
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

from sra_language_models import MoESRALanguageModel
from sra_experiment import find_unshared_synapses

### 1. プラグインの取り外し (`pop_synapses`)
まずは、小規模なモデルを構築し、動的にシナプスを追加した後、それを `pop_synapses` で取り外す動作を確認します。

In [ ]:
# モデルの初期化
dim = 128
layers = 2
num_synapses = 4
k = 2
syn_hidden = 256
vocab_size = 1000

print("=== プラグインの取り外しデモ ===")
model = MoESRALanguageModel(vocab_size, dim, layers, num_synapses, k, syn_hidden)
print(f"[初期状態] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの追加 (Hot-Swap)
print("\nプラグインとしてシナプスを 2つ 追加します...")
model.add_synapses(2, freeze_base=True)
print(f"[追加後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

# シナプスの取り外し (Undo Hot-Swap)
print("\n追加したシナプスを 1つ 取り外します...")
model.pop_synapses(1)
print(f"[取り外し後] 搭載シナプス数: {model.blocks[0].num_synapses}")
print(f"  ルーターの埋め込みテンソル形状: {model.blocks[0].router.get_full_emb().shape}")

このように、`pop_synapses(N)` を呼び出すことで、末尾からN個のシナプステンソルが物理的に削除され、モデルの容量を復元できます。

### 2. 特定ドメインのパージ (`clear_synapses` と `find_unshared_synapses`)
次に、複数ドメインを学習したベースモデルの中から、「特定のドメイン（ここでは仮に `math`）でしか使われていない不要なシナプス」を自動検出し、ゼロクリアによって安全に無効化するプロセスを実証します。

In [ ]:
import random
tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

device = "cuda" if torch.cuda.is_available() else "cpu"

# 仮のデータセットとバッチ関数の準備（デモ用）
domains = ["text", "code", "math"]
data = {d: torch.randint(0, vocab_size, (1000,)) for d in domains}

def dummy_get_batch(data_dict, batch_size, seq_len, domain):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    d_data = data_dict[domain]
    for i in range(batch_size):
        start = random.randint(0, len(d_data) - seq_len - 1)
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
    return x, y

# もう少し大きめのモデルを準備
multi_model = MoESRALanguageModel(vocab_size, dim=128, layers=2, num_synapses=16, k=2, syn_hidden=256).to(device)

In [ ]:
print("=== 特定ドメインのパージデモ ===")
print("Mathドメインでのみ使用され、他(Text, Code)と共用されていないシナプスを検索します...")

# ユーティリティを用いて対象シナプスを特定
unshared_synapses = find_unshared_synapses(
    model=multi_model, 
    data_dict=data, 
    target_domain="math", 
    other_domains=["text", "code"], 
    get_batch_func=dummy_get_batch,
    max_seq_len=32,
    threshold=0.01  # 1%以上の頻度で利用されていれば「使用している」と判定
)

print(f"\n抽出されたMath専用のシナプスインデックス: {unshared_synapses}")

※ 上記のインデックスは、未学習モデルの場合はランダムに分布するため見つからないことがあります。

抽出されたインデックス（見つからなかった場合はデモ用に適当なインデックス）を `clear_synapses` に渡し、該当シナプスをゼロクリアします。

In [ ]:
if len(unshared_synapses) > 0:
    target_idx = unshared_synapses[0]
else:
    print("※未学習モデルのため条件に完全合致するものがありませんでした。デモとしてシナプス 3 を対象にします。")
    unshared_synapses = [3]
    target_idx = 3

# クリア前の重みのノルムを確認
pre_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
pre_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア前] シナプス {target_idx} の Router埋め込みノルム: {pre_emb_norm:.4f}, W1ノルム: {pre_w1_norm:.4f}")

# ゼロクリア実行
multi_model.clear_synapses(unshared_synapses)
print("\nゼロクリアを実行しました。\n")

# クリア後の重みのノルムを確認
post_emb_norm = torch.norm(multi_model.blocks[0].router.get_full_emb()[target_idx]).item()
post_w1_norm = torch.norm(multi_model.blocks[0].get_full_param('w1')[target_idx]).item()
print(f"[クリア後] シナプス {target_idx} の Router埋め込みノルム: {post_emb_norm:.4f}, W1ノルム: {post_w1_norm:.4f}")

### 結論
`clear_synapses` は指定されたインデックスの重みを物理的に削除せず `0.0` にクリアします。
これにより、他のシナプスのインデックス（ID）がズレることを防ぎ、既存のメタデータマスクとの互換性を保ちながら、不要な計算パスを完全に無効化できます。空きスロットとなったインデックスには、後から新しいプラグインを上書き（Hot-Swap）することが可能です。